# Reproducing the robustness figure

The 3×3 figure summarising how much of the published analysis survives
resampling of the data. It is drawn from `results/perturbations.jld2`, a ~200 KB
committed artifact holding the per-mode energies of the 369 designed candidates
under each refit — so this notebook needs no training and runs in seconds.

Regenerate that artifact with `scripts/03_perturbations.jl` (hours on a laptop).

**What the figure says.** The energy *rankings* are robust and the absolute
*labels* are not: at 25 % of the reads the Spearman correlation of the energies
against the published model is still ~0.85, while only ~30 % of the specificity
calls survive. A fixed cutoff turns a small systematic shift into a changed
call, so the fragile conclusions are exactly the threshold-based ones.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using KrasPeptidePhageML, CairoMakie, DataFrames, Printf, Statistics
using StatsBase: corspearman

## The artifact

In [ ]:
(; sequence, E_reference, runs) = load_perturbations()
@printf("%d candidates, %d runs\n", length(sequence), length(runs))
combine(groupby(DataFrame(kind = [r.kind for r in runs], frac = [r.frac for r in runs]),
                [:kind, :frac]), nrow => :n_runs)

## The numbers behind the headline

Read depth against the two things it can break: the labels, and the ranking.

In [ ]:
ref = published_labels(E_reference)
base = Float64.(E_reference[:, MODES[:wt]]) .- Float64.(E_reference[:, MODES[:mutation]])
for f in (1.0, 0.5, 0.25)
    rs = filter(r -> r.kind == "resample" && r.frac == f, runs)
    isempty(rs) && continue
    lab = mean(mean(published_labels(r.E) .== ref) for r in rs)
    rho = mean(corspearman(Float64.(E_reference[:, MODES[:mutation]]),
                           Float64.(r.E[:, MODES[:mutation]])) for r in rs)
    jac = mean(topn_overlap([base, Float64.(r.E[:, MODES[:wt]]) .-
                                   Float64.(r.E[:, MODES[:mutation]])], 100)[1, 2] for r in rs)
    @printf("%3.0f%% reads (n=%2d):  labels %5.1f%%   Spearman E[mut] %.3f   top-100 Jaccard %.2f\n",
            100f, length(rs), 100lab, rho, jac)
end

Under cross-validation, per published class — the mutant-specific calls hold up
considerably better than the cross-specific ones.

In [ ]:
folds = filter(r -> r.kind == "cv", runs)
ag = label_agreement(E_reference, [r.E for r in folds]; sequences = sequence)
combine(groupby(ag, :published),
        nrow => :n,
        :frac_agree => mean => :mean_agreement,
        :n_agree => (v -> count(==(length(folds)), v)) => :all_folds)

## The figure

In [ ]:
fig = figure_combined(topn = 100,
                      outfile = joinpath(@__DIR__, "..", "figures", "figure_combined"))
fig

The SVG is vector throughout, including the text. Note one deviation from the
original matplotlib figure: that one set `svg.fonttype = "none"` and emitted real
`<text>` elements, editable as text in Illustrator or Inkscape. Cairo's SVG
surface always subsets glyphs into reusable outline definitions (`<g id="glyph-…">`
referenced by `<use>`), so with CairoMakie the labels are vector shapes, not
editable strings. Edit the labels here and re-run instead.

In [ ]:
svg = joinpath(@__DIR__, "..", "figures", "figure_combined.svg")
s = read(svg, String)
@printf("%s  %.0f KB\n", basename(svg), filesize(svg) / 1024)
@printf("  <text> elements     : %d   (Cairo emits none — see above)\n",
        count(_ -> true, eachmatch(r"<text", s)))
@printf("  glyph outline defs  : %d, referenced %d times\n",
        count(_ -> true, eachmatch(r"id=\"glyph", s)),
        count(_ -> true, eachmatch(r"<use ", s)))
@printf("  raster <image> tags : %d   (the two heatmap panels)\n",
        count(_ -> true, eachmatch(r"<image", s)))